# CAPE-TTA V9：最低算力 pilot

选择免费 **T4**，点击“全部运行”，授权本人的 Google Drive。默认只做 Spatial 两个 task、每task三个 init、K=4、H=12、单seed；**不再运行 V8 ALL**。

先一次模型加载保存 frozen/action-candidate bank；四个候选不调用7B模型做各自的闭环续接。CPU用共享 factual action tape评估候选，严格标为 **open-loop action-consequence**。

随后尝试rank2、单步、非Oracle target的真实LoRA，生成该更新分支自身的闭环短轨迹；CPU fresh replay验证frozen/adapted。更新失败保留bank和失败日志，不会伪装成adaptation结果。该更新路线尚未通过真实GPU集成测试，可能因T4显存不足停止。

每个完成episode直接保存Drive；同一固定版本再次运行会跳过完整bank。HF模型下载缓存复用当前运行时的 `/content/cape-hf-cache`；运行时重置后本地模型缓存可能丢失。不会自动占用Drive存储整个7B模型，也不会购买资源。失败的adapter不会自动重试。

**统计限制：** 六个独立episode划分2/2/2，alpha=.1时conformal q必为+infinity，LCB=-infinity，只能保守拒绝。这并不意味着CAPE控制住了HAR，也不证明CAPE无效。先观察真实参数收益和可学习性，再决定是否补独立calibration样本。未接受任何adaptation时HAR为null。

运行结束生成 `CAPE_TTA_V9_Minimal.zip`，请把完整ZIP发回本对话，才能由已授权的GitHub连接归档新结果到 `results/`。Colab自身没有继承本对话的GitHub写入凭据，不应在公开notebook中粘贴token。上传前的Drive结果尚未同步GitHub。

已有真实结果与本次CPU核验已归档到仓库 `results/verified/`，本轮资源阻塞见 `results/execution-20261030/GO_NO_GO.md`。完整协议见 `docs/V9_MINIMUM_COMPUTE_PROTOCOL.md`。


In [ ]:
#@title V9 最小算力 pilot（2 task × 3 init；K=4；H=12；单 seed）
SAVE_TO_DRIVE = True #@param {type:"boolean"}
ATTEMPT_REAL_LORA = True #@param {type:"boolean"}
# False 只做 action-consequence，不会产生 parameter-adaptation benefit。
import os, sys, json, subprocess, traceback, shutil, datetime, hashlib
from pathlib import Path
from google.colab import drive, files
SOURCE = 'ef1e31dcb0a3bd2815146344b06bf3dcb816bac7'
REPO = Path('/content/CAPE-TTA-V9-' + SOURCE[:12])
if SAVE_TO_DRIVE:
    drive.mount('/content/drive')
    ROOT = Path('/content/drive/MyDrive/CAPE_TTA_V9_Minimal')
else:
    ROOT = Path('/content/CAPE_TTA_V9_Minimal')
ROOT.mkdir(parents=True, exist_ok=True)
PYTHON = Path('/content/cape-env-v3/bin/python')
LIBERO = Path('/content/LIBERO-8f1084e3132a')
status = {'status':'running','source_commit':SOURCE,'started_utc':datetime.datetime.now(datetime.timezone.utc).isoformat(),
          'requested_adaptation':ATTEMPT_REAL_LORA,'paid_resources_allowed':False}

def record():
    tmp = ROOT/'launcher_status.json.tmp'
    tmp.write_text(json.dumps(status,indent=2)+'\n'); tmp.replace(ROOT/'launcher_status.json')

def run(command, *, env=None, check=True):
    print('$', ' '.join(map(str,command)),flush=True)
    with (ROOT/'launcher.log').open('a') as log:
        log.write('$ '+' '.join(map(str,command))+'\n');log.flush()
        with subprocess.Popen(list(map(str,command)),stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1,env=env) as proc:
            try:
                for line in proc.stdout:
                    print(line,end='',flush=True);log.write(line);log.flush()
                rc=proc.wait()
            except BaseException:
                proc.terminate()
                try: proc.wait(timeout=10)
                except subprocess.TimeoutExpired: proc.kill();proc.wait()
                raise
    if check and rc: raise subprocess.CalledProcessError(rc,command)
    return rc

try:
    record()
    if not REPO.exists():
        run(['git','clone','https://github.com/weichezhang/CAPE-TTA.git',REPO])
        run(['git','-C',REPO,'checkout','--detach',SOURCE])
    actual=subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'],text=True).strip()
    if actual!=SOURCE: raise RuntimeError('拒绝使用不一致的源代码版本')
    config=json.loads((REPO/'configs/pilot/v9_minimal.json').read_text())
    expected=[f'spatial-task-{t}-init-{s}-seed-{config["seed"]}' for t in config['task_ids'] for s in config['init_state_ids']]
    bank_paths=[ROOT/'banks'/f'{key}.json' for key in expected]
    def checked_bank(path):
        if not path.exists(): return None
        checksum=Path(str(path)+'.sha256.json')
        if not checksum.exists(): raise RuntimeError('动作bank缺少校验文件；保留文件并停止')
        if hashlib.sha256(path.read_bytes()).hexdigest()!=json.loads(checksum.read_text())['sha256']:
            raise RuntimeError('动作bank校验失败；保留文件并停止')
        return json.loads(path.read_text())
    banks=[checked_bank(p) for p in bank_paths]
    for b in banks:
        if b and b['runtime']['source_commit']!=SOURCE: raise RuntimeError('缓存源版本不一致；不混合实验')
        if b and ATTEMPT_REAL_LORA and b.get('adaptation',{}).get('status')=='failed':
            raise RuntimeError('之前LoRA失败，已保留；请先检查gpu.log，不自动重试或扩大实验')
    gpu_needed=not all(b and b['status']=='bank_complete' and (not ATTEMPT_REAL_LORA or b.get('adaptation',{}).get('status')=='complete') for b in banks)
    print('已保存动作bank：',sum(b is not None for b in banks),'/',len(expected),'需要GPU采集：',gpu_needed,flush=True)
    if gpu_needed:
        if shutil.which('nvidia-smi') is None: raise RuntimeError('请选择免费的T4运行时。不会改用付费GPU。')
        run(['nvidia-smi','--query-gpu=name,memory.total,driver_version','--format=csv,noheader'])
    env_ready=False
    if PYTHON.exists() and LIBERO.exists():
        probe='import torch,transformers,peft,mujoco; assert torch.__version__.startswith("2.2.0"); assert transformers.__version__=="4.40.1"; assert peft.__version__=="0.11.1"'
        env_ready=subprocess.run([str(PYTHON),'-c',probe],capture_output=True).returncode==0
    if not env_ready:
        # Existing tested bootstrap is GPU-specific. No GPU means stop, not a paid fallback.
        if not gpu_needed: raise RuntimeError('bank已完整，但本运行时没有CPU模拟器依赖环境；保留bank供后续CPU评估')
        setup=ROOT/('setup-'+datetime.datetime.now().strftime('%Y%m%d-%H%M%S'))
        run([sys.executable,'-u',REPO/'scripts/colab_bootstrap.py','--prepare-only','--run-dir',setup])
    else:
        print('复用现有固定版本环境，跳过重新安装依赖。',flush=True)
        run([PYTHON,'-m','pip','install','--no-build-isolation','--no-deps','-e',REPO,'-e',LIBERO])
    libero_sha=subprocess.check_output(['git','-C',str(LIBERO),'rev-parse','HEAD'],text=True).strip()
    if libero_sha!='8f1084e3132a39270c3a13ebe37270a43ece2a01': raise RuntimeError('LIBERO版本不一致')
    env=dict(os.environ,HF_HOME='/content/cape-hf-cache',MUJOCO_GL='osmesa',PYOPENGL_PLATFORM='osmesa',
             TOKENIZERS_PARALLELISM='false',TF_CPP_MIN_LOG_LEVEL='2',PYTHONUNBUFFERED='1')
    gpu_rc=0
    if gpu_needed:
        status['stage']='gpu_bank_collection';record()
        gpu_rc=run([PYTHON,'-u',REPO/'scripts/build_cached_pilot.py','--libero-root',LIBERO,'--bank-root',ROOT,
                    '--adaptation','candidate-self-distill' if ATTEMPT_REAL_LORA else 'none'],env=env,check=False)
    # A failed update still leaves sealed base banks available for CPU diagnostics.
    if not any((ROOT/'banks').glob('*.json')): raise RuntimeError('还没有完整动作bank；查看安装/GPU失败日志')
    status['stage']='cpu_fresh_replay';status['gpu_return_code']=gpu_rc;record()
    cpu_rc=run([PYTHON,'-u',REPO/'scripts/evaluate_cached_pilot.py','--libero-root',LIBERO,'--bank-root',ROOT,'--alpha','0.1'],
               env=dict(env,CUDA_VISIBLE_DEVICES=''),check=False)
    status['cpu_return_code']=cpu_rc
    status['status']='pilot_execution_complete' if gpu_rc==0 and cpu_rc==0 else 'partial_or_failed_preserved'
    report_path=ROOT/'cpu/report.json'
    if report_path.exists():
        report=json.loads(report_path.read_text())
        status['go_no_go']=report.get('benefit_gate',{}).get('status','NO_GO_FOR_EXPANSION')
        print('实际完成CPU episode数：',len(report['episodes']),flush=True)
        print('GO/NO-GO：',json.dumps(report.get('benefit_gate',{}),ensure_ascii=False,indent=2),flush=True)
    print('本格不会自动扩展suite或多seed。',flush=True)
except BaseException:
    status.update(status='failed',traceback=traceback.format_exc());traceback.print_exc()
finally:
    record()
    bundle=shutil.make_archive('/content/CAPE_TTA_V9_Minimal','zip',ROOT)
    print('已保留状态：',status['status'],'；结果包：',bundle,flush=True)
    if SAVE_TO_DRIVE: print('动作bank与失败日志已逐episode直接保存至：',ROOT,flush=True)
    try: files.download(bundle)
    except Exception as exc: print('自动下载不可用；请从文件面板下载ZIP：',exc,flush=True)
